# internsg.com Search

Collects the internship posts on [internsg.com](https://www.internsg.com) that are still open, finds an application e-mail address for each company, and saves everything to an Excel job list.

## Production Code
##### This code runs in a virtual environment using Visual Studio Code.

Run the cells from top to bottom: **1. install packages → 2. imports → 3. settings → 4. functions → 5. run**.

### 1. Install the packages (run once)
This installs every package the notebook imports. It is the same as typing `pip install -r requirements.txt` in the terminal.

In [ ]:
%pip install -r requirements.txt

### 2. Imports

In [ ]:
import codecs
import csv
import os
import re
import time
from datetime import date
from urllib.parse import quote_plus, unquote, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook, load_workbook
from selenium import webdriver
from selenium.common.exceptions import (InvalidArgumentException, NoSuchElementException,
                                        StaleElementReferenceException, TimeoutException,
                                        WebDriverException)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

### 3. Settings
Change the values in this cell, then run it.

In [ ]:
START_PAGE = 1     # First listing page to scrape (page 1 shows the newest posts)
END_PAGE = 2       # Last listing page to scrape

# The Excel file that the results are saved to.
# If the file already exists, new records are added at the bottom (posts that are already in the file are skipped).
TODAY = date.today()
OUTPUT_FILE = f"Joblist updated to {TODAY.year}.{TODAY.month}.{TODAY.day}.xlsx"

PAGE_LOAD_WAIT = 4         # Seconds to wait after a page opens in Chrome (smaller = faster, larger = safer)
HEADLESS = True            # True = Chrome runs invisibly (faster); False = you can watch the browser work
USE_GOOGLE_SEARCH = True   # Search Google when the company's own website shows no e-mail address
MAX_PAGES_PER_COMPANY = 8  # Most pages that are opened while looking for the e-mail of one company

# Only needed if Chrome or its driver cannot be found automatically (see the README, "Chrome driver")
CHROMEDRIVER_PATH = None   # For example: r"C:\chromedriver-win64\chromedriver.exe"
CHROME_BINARY = None       # For example: r"C:\Program Files\Google\Chrome\Application\chrome.exe"

# Advanced: there is normally no need to change these
BASE_URL = 'https://www.internsg.com'
LISTING_URL = BASE_URL + '/jobs/{page}/?f_p&f_i&filter_s#isg-top'
SEARCH_URL = 'https://www.google.com/search?q={query}'
PAGE_TIMEOUT = 30          # Seconds before Chrome gives up on a page that does not finish loading

### 4. Functions
#### Downloading pages and reading e-mail addresses

In [ ]:
COLUMNS = ['Company', 'Position', 'E-mail', 'E-mail List', 'Domain', 'Date', 'Link']

# Job-related words that may appear in the first part of an e-mail address, from most to least relevant
EMAIL_KEYWORDS = ['careers', 'career', 'recruit', 'hiring', 'jobs', 'job', 'talent', 'internship', 'hr',
                  'contact', 'info', 'admin', 'customerservice', 'hello', 'hi', 'support', 'help', 'sales',
                  'marketing']

EMAIL_REGEX = re.compile(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)*\.[A-Za-z]{2,}')
DOMAIN_REGEX = re.compile(r'^(?:[a-z0-9](?:[a-z0-9\-]*[a-z0-9])?\.)+[a-z]{2,}$')
INTERN_REGEX = re.compile(r'\bintern(?:ship)?s?\b', re.IGNORECASE)   # 'Intern', 'Internship' but not 'International'
DATE_REGEX = re.compile(r'\b\d{1,2}\s*(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\b', re.IGNORECASE)

# Things that look like e-mail addresses but are not (image names such as logo@2x.png, sample or no-reply addresses)
FILE_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif', 'svg', 'webp', 'ico', 'css', 'js', 'woff', 'woff2', 'ttf', 'eot'}
NOT_REAL_NAMES = {'example', 'yourname', 'your.name', 'name', 'email', 'user', 'username', 'test', 'janedoe',
                  'johndoe', 'jane.doe', 'john.doe', 'firstname.lastname', 'noreply', 'no-reply', 'donotreply',
                  'do-not-reply', 'mailer-daemon', 'postmaster'}

# Words in a link that suggest a page with contact details (best first), and sites that are not worth opening
SITE_PAGE_WORDS = ['contact', 'career', 'job', 'recruit', 'join', 'enquir', 'inquir', 'about', 'privacy', 'team']
FILE_LINK_ENDINGS = ('.pdf', '.jpg', '.jpeg', '.png', '.gif', '.svg', '.zip', '.doc', '.docx', '.xls', '.xlsx',
                     '.ppt', '.pptx', '.mp4')
SKIPPED_SITES = ('facebook.com', 'instagram.com', 'linkedin.com', 'youtube.com', 'twitter.com', 'x.com',
                 'tiktok.com')

# Pretend to be a normal browser, because some websites refuse the default python-requests client
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
})


def download(url, retries=3, timeout=20):
    # Download a URL. Returns the response, or None if the website could not be reached
    for attempt in range(1, retries + 1):
        try:
            response = SESSION.get(url, timeout=timeout)
        except requests.RequestException as error:
            print(f"  Download failed ({attempt}/{retries}): {type(error).__name__}")
        else:
            # Errors like 403 or 404 will not go away by trying again, but server errors (5xx) might
            if response.status_code < 500 and response.status_code != 429:
                return response
            print(f"  HTTP {response.status_code} ({attempt}/{retries}): {url}")
        if attempt < retries:
            time.sleep(2 * attempt)
    return None


def fetch_html(url, retries=3, timeout=20):
    # Download a web page. Returns the HTML, or None if the page is missing, blocked or not a web page
    response = download(url, retries, timeout)
    if response is None or response.status_code != 200:
        return None
    content_type = response.headers.get('Content-Type', '').lower()
    if content_type and 'html' not in content_type and 'xml' not in content_type:
        return None  # A PDF, an image, ...
    return response.content


def clean_link(href, base_url=None):
    # Turn a link into a full http(s) web address. Returns None for mailto:, tel:, javascript: and empty links
    if not href or href.strip().startswith('#'):
        return None
    url = urljoin(base_url or BASE_URL, href.strip())
    parts = urlparse(url)
    return url if parts.scheme in ('http', 'https') and parts.netloc else None


def clean_domain(text):
    # 'https://www.Example.com/about' -> 'example.com'. Returns None if the text does not look like a domain
    if not text:
        return None
    text = re.sub(r'^[a-z]+://', '', text.strip().lower())
    text = re.split(r'[/?#\s]', text)[0]
    if text.startswith('www.'):
        text = text[4:]
    return text if DOMAIN_REGEX.match(text) else None


def email_matches_domain(email, domain):
    # True if the e-mail belongs to the domain. Sub-domains such as sg.company.com are accepted as well
    host = email.rsplit('@', 1)[-1].lower()
    return host == domain or host.endswith('.' + domain)


def host_on_domain(url, domain):
    host = (urlparse(url).hostname or '').lower()
    return host == domain or host.endswith('.' + domain)


def is_skipped_site(url):
    # True for Google's own pages and for sites that need a login (not worth opening)
    host = (urlparse(url).hostname or '').lower()
    return (bool(re.search(r'(^|\.)google(usercontent)?\.', host))
            or any(host == site or host.endswith('.' + site) for site in SKIPPED_SITES))


def clean_email(email):
    # Tidy up a found e-mail address. Returns None if it is not a real address
    email = email.strip().strip('.-_').lower()
    name, _, host = email.partition('@')
    if not name or not host or '..' in email:
        return None
    if host.rsplit('.', 1)[-1] in FILE_EXTENSIONS or name in NOT_REAL_NAMES:
        return None
    return email


def find_emails(text):
    # Return every e-mail address in a piece of text (lower case, no duplicates, in order of appearance)
    # Some pages write addresses as 'name [at] company [dot] com'
    text = re.sub(r'\s*[\[({]\s*at\s*[\])}]\s*', '@', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*[\[({]\s*dot\s*[\])}]\s*', '.', text, flags=re.IGNORECASE)
    emails = []
    # Only look at the words that contain an '@' (much faster than searching the whole text)
    for word in re.split(r'[\s<>"\'(){}\[\],;:|]+', text):
        if '@' not in word or len(word) > 254:
            continue
        for match in EMAIL_REGEX.findall(word):
            email = clean_email(match)
            if email and email not in emails:
                emails.append(email)
    return emails


def decode_enc_email(value):
    # InternSG hides e-mail addresses from robots: the page holds a scrambled copy in a data-enc-email attribute.
    # The scramble is ROT13, with '[at]' written instead of '@'. Returns the real address, or None
    if '@' in value:
        return clean_email(value)  # Not scrambled after all
    value = value.replace('[at]', '@').replace('[ng]', '@')  # '[ng]' is how ROT13 writes '[at]'
    found = find_emails(codecs.decode(value, 'rot_13'))
    return found[0] if found else None


def decode_cf_email(value):
    # Many websites use Cloudflare to hide e-mail addresses as a hex code: the first byte is a key, and every
    # other byte is one letter of the address combined (XOR) with that key. Returns the real address, or None
    try:
        key = int(value[:2], 16)
        address = ''.join(chr(int(value[i:i + 2], 16) ^ key) for i in range(2, len(value), 2))
    except ValueError:
        return None
    found = find_emails(address)
    return found[0] if found else None


def emails_from_tag(tag):
    # Collect every e-mail address in a web page (or one part of it), including the hidden ones
    found = []
    # 1. Addresses hidden by the website's protection. They are decoded here, so no browser is needed
    for element in tag.find_all(attrs={'data-enc-email': True}):
        found.append(decode_enc_email(element['data-enc-email']))
    for element in tag.find_all(attrs={'data-cfemail': True}):
        found.append(decode_cf_email(element['data-cfemail']))
    # 2. mailto: links
    for link in tag.find_all('a', href=True):
        href = link['href'].strip()
        if href.lower().startswith('mailto:'):
            found.extend(find_emails(unquote(href[7:]).split('?')[0]))
        elif 'email-protection#' in href:
            found.append(decode_cf_email(href.split('#', 1)[1]))
    # 3. The text of the page
    found.extend(find_emails(tag.get_text(' ', strip=True)))

    emails = []
    for email in found:
        if email and email not in emails:
            emails.append(email)
    return emails


def company_emails(emails, domain):
    # Keep only the e-mails that belong to the company's domain
    return [email for email in emails if email_matches_domain(email, domain)]

#### Chrome (used when a page needs JavaScript, and for the Google search)

In [ ]:
def create_chrome_driver(headless=True):
    # Set up Chrome options
    chrome_options = Options()
    if headless:  # Running in the headless mode will be faster
        chrome_options.add_argument("--headless=new")

    # Other options
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})  # No pictures = faster
    if CHROME_BINARY:
        chrome_options.binary_location = CHROME_BINARY

    # Selenium downloads the matching Chrome driver by itself, unless CHROMEDRIVER_PATH is set
    service = Service(executable_path=CHROMEDRIVER_PATH) if CHROMEDRIVER_PATH else Service()
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(PAGE_TIMEOUT)

    # Return the Chrome Driver
    return driver


class BrowserSession:
    # One Chrome window that is re-used for the whole run (much faster than starting Chrome for every page).
    # If Chrome cannot be started, the scraper still works: it just skips the steps that need a browser.

    def __init__(self):
        self.driver = None
        self.available = True        # Becomes False if Chrome cannot be started
        self.search_blocked = False  # Becomes True if Google shows a robot check

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

    def start(self):
        try:
            self.driver = create_chrome_driver(headless=HEADLESS)
        except Exception as error:
            self.driver = None
            self.available = False
            print("Chrome could not be started, so pages that need a browser (and the Google search) are skipped.")
            print("  Reason:", str(error).strip().splitlines()[0] if str(error).strip() else type(error).__name__)
            print("  See the README, section 'Chrome driver', for how to fix this.")

    def open(self, url):
        # Open a page. Returns True if the page was loaded (or at least partly loaded)
        if not url.lower().startswith(('http://', 'https://')):
            return False
        for attempt in (1, 2):
            if self.driver is None and self.available:
                self.start()
            if self.driver is None:
                return False
            try:
                self.driver.get(url)
                return True
            except TimeoutException:
                try:
                    self.driver.execute_script('window.stop();')  # Use whatever has loaded so far
                except WebDriverException:
                    pass
                return True
            except InvalidArgumentException:
                return False
            except WebDriverException as error:
                print("  Chrome stopped working, restarting it:", type(error).__name__)
                self.close()
        return False

    def page_source(self):
        try:
            return self.driver.page_source if self.driver else ''
        except WebDriverException:
            return ''

    def current_url(self):
        try:
            return self.driver.current_url if self.driver else ''
        except WebDriverException:
            return ''

    def find_elements(self, by, value):
        try:
            return self.driver.find_elements(by, value) if self.driver else []
        except WebDriverException:
            return []

    def get_soup(self, ready=None, timeout=10):
        # Read the page that is open. If ready() is given, keep checking (up to `timeout` seconds) until
        # ready(soup) is True, for pages that fill themselves in with JavaScript
        deadline = time.time() + timeout
        while True:
            soup = BeautifulSoup(self.page_source(), 'html.parser')
            if ready is None or ready(soup) or time.time() >= deadline:
                return soup
            time.sleep(0.5)

    def looks_like_robot_check(self):
        # True if Google shows its 'unusual traffic' page instead of search results
        return ('/sorry/' in self.current_url().lower()
                or 'unusual traffic from your computer network' in self.page_source()[:200000].lower())

    def close(self):
        if self.driver is not None:
            try:
                self.driver.quit()
            except WebDriverException:
                pass
        self.driver = None

#### Reading the InternSG pages

In [ ]:
def get_labeled_value(soup, label):
    # A job page shows its details as rows of 'label + value', for example
    # <div class="ast-col-md-2 font-weight-bold">Company</div> followed by a <div> with the value.
    # Returns the value that belongs to the label, or None if the label is not on the page
    wanted = label.lower()
    # Look in the layout that InternSG uses first, then (in case the layout changes a little) in any small tag
    for tags in (soup.find_all('div', class_='font-weight-bold'),
                 soup.find_all(['div', 'dt', 'th', 'td', 'strong', 'b', 'span', 'p'])):
        for tag in tags:
            if ' '.join(tag.get_text().split()).lower() == wanted:
                value = tag.find_next_sibling()
                if value is not None:
                    return value
    return None


def split_company_and_domain(tag):
    # A company cell holds the company name and, in a small badge, its web domain. Returns (name, domain)
    domain, badge = None, None
    for element in tag.find_all('span') + tag.find_all(['code', 'kbd', 'mark', 'small', 'b', 'strong', 'em', 'a']):
        domain = clean_domain(element.get_text(strip=True))
        if domain:
            badge = element
            break
    name_parts = [text.strip() for text in tag.find_all(string=True)
                  if text.strip() and not any(parent is badge for parent in text.parents)]
    return (' '.join(name_parts) or None), domain


def company_name_from_title(detail_soup):
    # Backup plan: the page title looks like 'Company Name - Position - InternSG'
    title_tag = detail_soup.find('title')
    if title_tag:
        title_text = title_tag.get_text(strip=True)
        for separator in (' - ', ' – ', ' — ', ' | '):
            if separator in title_text:
                return title_text.split(separator)[0].strip()
    return None


def extract_company_name(detail_soup):
    # Extract the company name and its domain from the 'Company' section
    company_name, domain = None, None
    company_div = get_labeled_value(detail_soup, 'Company')
    if company_div is not None:
        company_name, domain = split_company_and_domain(company_div)
    else:
        print("No company section found.")

    if not company_name:
        company_name = company_name_from_title(detail_soup)

    print('Company name:', company_name)
    print('Domain:', domain)
    return company_name, domain


def extract_contact_info(detail_soup):
    # Check the 'Application Instructions' section.
    # Returns the e-mail address given there (or None) and the list of links given there
    email = None
    contact_links = []
    instructions = get_labeled_value(detail_soup, 'Application Instructions')
    if instructions is None:
        print("No 'Application Instructions' section found.")
        return email, contact_links

    # Find the e-mail address provided in the 'Application Instructions' section
    emails = emails_from_tag(instructions)
    if emails:
        email = choose_best_email(emails) or emails[0]

    # Find all the links provided in the 'Application Instructions' section
    for link in instructions.find_all('a', href=True):
        url = clean_link(link['href'])
        if url and url not in contact_links:
            contact_links.append(url)

    print('Email:', email)
    if contact_links:
        print('Links in the instructions:', contact_links)
    return email, contact_links


def has_job_details(soup):
    # True if the page really is a job page (it has the 'Company' row)
    return get_labeled_value(soup, 'Company') is not None


def email_is_hidden(detail_soup):
    # True if the job page has an e-mail address that we could not read (it only shows up when JavaScript runs)
    instructions = get_labeled_value(detail_soup, 'Application Instructions')
    if instructions is None:
        return False
    text = instructions.get_text(' ').lower()
    hidden = bool(instructions.find(attrs={'data-enc-email': True}) or instructions.find(attrs={'data-cfemail': True})
                  or 'email protected' in text or 'protected email' in text)
    return hidden and not emails_from_tag(instructions)


def find_status_div(listing):
    # The 'Post' column of a listing row: it shows the post date, or 'Closed'
    candidates = listing.find_all('div', class_='ast-col-lg-1')
    for div in candidates:
        text = div.get_text(strip=True)
        if text.lower() == 'closed' or DATE_REGEX.search(text):
            return div
    return candidates[0] if candidates else None


def get_open_internships(page_num):
    # Read one listing page. Returns (number of job rows on the page, list of posts to process).
    # A post is processed if it is still open and its title contains the word 'intern'.
    # Returns (None, []) if the page could not be loaded
    response = download(LISTING_URL.format(page=page_num))
    if response is None:
        return None, []
    if response.status_code == 404:
        return 0, []  # This page does not exist
    if response.status_code != 200:
        print(f"Listing page {page_num}: HTTP {response.status_code}")
        return None, []

    soup = BeautifulSoup(response.content, 'html.parser')
    rows_seen = 0
    postings = []

    # Find all internship listings on the page
    for listing in soup.find_all('div', class_='ast-row'):
        if listing.find('div', class_='ast-row'):
            continue  # This is a box around several rows, not a job

        # Ensure the company div exists before accessing it
        company_div = listing.find('div', class_='ast-col-lg-3')
        if not company_div:
            continue  # Skip if company div is not found

        position_div = company_div.find_next_sibling('div')
        if not position_div:
            continue  # Skip if position div is not found

        # Extract the link to the detailed internship page
        detail_link_tag = position_div.find('a', href=True)
        if not detail_link_tag:
            continue  # Skip if detail link is not found
        detail_link = clean_link(detail_link_tag['href'])
        if not detail_link:
            continue
        rows_seen += 1

        # Extract the post status
        post_status_div = find_status_div(listing)
        if not post_status_div:
            continue  # Skip if post status div is not found
        post_status = post_status_div.get_text(strip=True)

        # Only proceed if the post is not closed
        if 'closed' in post_status.lower():
            continue

        position_title = detail_link_tag.get_text(strip=True) or position_div.get_text(strip=True)
        if not INTERN_REGEX.search(position_title):  # Check if title contains the word 'intern'
            continue

        date_span = post_status_div.find('span')  # Get the post date
        date_text = (date_span or post_status_div).get_text(strip=True)
        date_match = DATE_REGEX.search(date_text)
        company, domain = split_company_and_domain(company_div)
        postings.append({
            'title': position_title,
            'date': date_match.group(0) if date_match else date_text,
            'link': detail_link,
            'company': company,   # The listing also shows the company and its domain (used if the job page fails)
            'domain': domain,
        })
    return rows_seen, postings


def get_job_page(detail_link, browser):
    # Open the job description page. Returns the page as a BeautifulSoup object (or None if it cannot be loaded).
    # A quick download is tried first; Chrome is only used if that is not enough
    soup = None
    html = fetch_html(detail_link)
    if html:
        soup = BeautifulSoup(html, 'html.parser')
        if has_job_details(soup) and not email_is_hidden(soup):
            return soup

    if browser.available:
        # Try up to three times (a page sometimes comes back empty the first time)
        for attempt in range(1, 4):
            if browser.open(detail_link):
                browser_soup = browser.get_soup(
                    ready=lambda page: has_job_details(page) and not email_is_hidden(page),
                    timeout=PAGE_LOAD_WAIT + 6)
                if has_job_details(browser_soup):
                    return browser_soup
            print(f"  The job page is not ready, trying again ({attempt}/3)")
            time.sleep(1)
    return soup

#### Finding the e-mail address of a company

In [ ]:
def keyword_in_name(keyword, name):
    # True if the keyword appears in the first part of an e-mail address
    words = re.split(r'[^a-z]+', name.lower())  # 'hr-sg' -> ['hr', 'sg']
    if len(keyword) <= 3:
        return keyword in words  # Short keywords must be a whole word, so 'chris' does not count as 'hr'
    return any(keyword in word for word in words)


def choose_best_email(email_addresses_list):
    # Given a list of email addresses, return the most relevant one based on predefined keywords
    # (the first keyword in EMAIL_KEYWORDS is the most relevant)
    emails = sorted(set(email_addresses_list))
    if not emails:
        return None

    # If the list has only one email, return it directly
    if len(emails) == 1:
        return emails[0]

    for keyword in EMAIL_KEYWORDS:
        matches = [email for email in emails if keyword_in_name(keyword, email.split('@')[0])]
        if matches:
            # If several emails match, take the shortest ('support@' rather than 'support.mumbai@')
            return min(matches, key=lambda email: (len(email.split('@')[0]), email))

    return None  # Return None if no relevant email is found


def find_site_links(soup, base_url, domain):
    # Links on a company website that probably lead to a contact, careers or about page (best first)
    ranked = []
    for tag in soup.find_all('a', href=True):
        url = clean_link(tag['href'], base_url)
        if not url:
            continue
        url = url.split('#')[0]
        if not host_on_domain(url, domain) or url.lower().endswith(FILE_LINK_ENDINGS):
            continue  # Stay on the company's own website
        text = f"{tag.get_text(' ', strip=True)} {url}".lower()
        for rank, word in enumerate(SITE_PAGE_WORDS):
            if word in text:
                ranked.append((rank, url))
                break
    ranked.sort(key=lambda item: item[0])

    links = []
    for _, url in ranked:
        if url not in links and url.rstrip('/') != base_url.rstrip('/'):
            links.append(url)
    return links


def search_company_website(domain, contact_links=()):
    # Look for e-mail addresses on the company's own website: the home page, its contact / careers / about
    # pages, and the links given in the job's Application Instructions. This only downloads pages (no browser)
    email_addresses = set()
    pages_read = 0

    home_url, home_soup = None, None
    for url in (f'https://{domain}/', f'https://www.{domain}/', f'http://{domain}/'):
        html = fetch_html(url, retries=1, timeout=10)
        if html:
            home_url, home_soup = url, BeautifulSoup(html, 'html.parser')
            break

    to_visit = []
    if home_soup is None:
        print(f"  Could not open the website of {domain}")
    else:
        pages_read += 1
        email_addresses.update(company_emails(emails_from_tag(home_soup), domain))
        to_visit = find_site_links(home_soup, home_url, domain)
        if not to_visit:  # No contact page was found in the menu, so try the usual addresses
            to_visit = [urljoin(home_url, path) for path in ('/contact', '/contact-us', '/about-us')]
    to_visit += [link for link in contact_links if link not in to_visit]

    for url in to_visit:
        if pages_read >= MAX_PAGES_PER_COMPANY:
            break
        time.sleep(0.3)  # Be gentle with the website
        html = fetch_html(url, retries=1, timeout=10)
        if not html:
            continue
        pages_read += 1
        print(f"  Visited {url}")
        email_addresses.update(company_emails(emails_from_tag(BeautifulSoup(html, 'html.parser')), domain))

    for email in sorted(email_addresses):
        print(f"  Found email on the company website: {email}")
    return email_addresses


def search_for_email(browser, company_name, domain, max_links=5):
    # Search Google for the company and read the e-mail addresses on the results page and on the most
    # promising result pages. Only addresses that belong to the company's domain are kept
    email_addresses = set()
    if not browser.available or browser.search_blocked:
        return email_addresses

    search_query = f"{company_name} {domain} contact email"
    if not browser.open(SEARCH_URL.format(query=quote_plus(search_query))):
        return email_addresses
    time.sleep(PAGE_LOAD_WAIT)  # Wait for the page to load

    if browser.looks_like_robot_check():
        browser.search_blocked = True
        print("  Google is asking for a robot check, so Google searches are skipped for the rest of this run.")
        return email_addresses

    # Extract emails from the search results page
    for email in company_emails(emails_from_tag(browser.get_soup()), domain):
        email_addresses.add(email)
        print(f"  Found email from search results page: {email}")

    # Grab search results
    contact_links = []
    first_links = []
    for result in browser.find_elements(By.CSS_SELECTOR, 'h3'):
        try:
            link = clean_link(result.find_element(By.XPATH, './ancestor::a[1]').get_attribute('href'))
            title = result.text
        except (NoSuchElementException, StaleElementReferenceException):
            continue
        if not link or is_skipped_site(link):
            continue
        first_links.append(link)
        # Filter links that contain "Contact", "About", or "Privacy"
        if any(word in f"{title} {link}".lower() for word in ('contact', 'about', 'privacy')):
            contact_links.append(link)
            print(f"  Found contact link: {link}")

    # Select first three results if no contact found
    links = contact_links or first_links[:3]
    links = sorted(links, key=lambda link: not host_on_domain(link, domain))[:max_links]  # The company's own pages first

    # Visit each contact link
    for link in links:
        print(f"  Visiting {link}")
        if not browser.open(link):
            continue
        time.sleep(PAGE_LOAD_WAIT)  # Wait for the page to load

        # Search for any email addresses on the page and check that they belong to the correct domain
        for email in company_emails(emails_from_tag(browser.get_soup()), domain):
            email_addresses.add(email)
            print(f"  Found email from page: {email}")

    return email_addresses


def find_company_emails(browser, company_name, domain, contact_links=()):
    # Look for the e-mail addresses of a company: first on its own website (quick),
    # and - only if nothing is found there - on Google
    email_addresses = set(search_company_website(domain, contact_links))
    if not email_addresses and USE_GOOGLE_SEARCH:
        email_addresses |= search_for_email(browser, company_name, domain)
    return sorted(email_addresses)

#### Saving the results

In [ ]:
def save_to_excel(data, filename=None):
    # Add records to the Excel file. The file (with its header row) is created if it does not exist yet.
    # Returns True if the records were saved, or False if they could not be saved
    filename = filename or OUTPUT_FILE
    for attempt in range(1, 7):
        try:
            file_exists = os.path.isfile(filename)
            if file_exists:
                # Append to the existing file
                workbook = load_workbook(filename)
                sheet = workbook['Sheet1'] if 'Sheet1' in workbook.sheetnames else workbook.active
            else:
                # Create a new file with headers if it doesn't exist
                workbook = Workbook()
                sheet = workbook.active
                sheet.title = 'Sheet1'
                sheet.append(COLUMNS)
                sheet.freeze_panes = 'A2'
                for column, width in zip('ABCDEFG', (36, 42, 34, 52, 26, 10, 70)):
                    sheet.column_dimensions[column].width = width

            for record in data:
                # A list cannot be stored in a cell, so the E-mail List is stored as text, like "['a@b.com']"
                sheet.append([str(value) if isinstance(value, (list, tuple, set)) else value for value in record])
                for cell in sheet[sheet.max_row]:
                    if cell.data_type == 'f':  # Text that starts with '=' must stay text, it is not a formula
                        cell.data_type = 's'

            workbook.save(filename)
            print(f"{'Data appended to' if file_exists else 'Initial data written to'} {filename}")
            return True
        except PermissionError:
            print(f"'{filename}' is open in another program (probably Excel). Please close it. "
                  f"Trying again in 5 seconds ({attempt}/6)...")
            time.sleep(5)
        except Exception as error:
            print(f"Could not save to '{filename}': {error}")
            return False
    return False


def save_backup_csv(data, filename=None):
    # If the Excel file cannot be saved, the records are kept in a CSV file next to it so nothing is lost
    backup_name = os.path.splitext(filename or OUTPUT_FILE)[0] + ' (unsaved records).csv'
    new_file = not os.path.isfile(backup_name)
    with open(backup_name, 'a', newline='', encoding='utf-8-sig') as backup_file:
        writer = csv.writer(backup_file)
        if new_file:
            writer.writerow(COLUMNS)
        writer.writerows([[str(value) if isinstance(value, (list, tuple, set)) else value for value in record]
                          for record in data])
    return backup_name


def load_saved_links(filename=None):
    # The links that are already in the Excel file, so a second run skips them instead of adding duplicates
    filename = filename or OUTPUT_FILE
    if not os.path.isfile(filename):
        return set()
    try:
        return set(pd.read_excel(filename, usecols=['Link'])['Link'].dropna().astype(str))
    except Exception as error:
        print(f"Could not read the existing links from '{filename}': {error}")
        return set()

#### The main function

In [ ]:
def build_record(browser, posting, email_cache):
    # Collect everything about one job post. Returns one record (one row of the Excel file)
    detail_soup = get_job_page(posting['link'], browser)
    if detail_soup is None:
        raise RuntimeError('the job page could not be loaded')

    # Obtain the company name and its domain (the job list is the backup if the job page does not show them)
    company_name, domain = extract_company_name(detail_soup)
    company_name = company_name or posting['company'] or 'Unknown'
    domain = domain or posting['domain']

    # Extract email
    best_email = None
    email_addresses_list = []
    email, contact_links = extract_contact_info(detail_soup)
    if email:
        best_email = email
    elif not domain:
        print("No domain is known for this company, so no e-mail search is possible.")
    else:
        # If no email is found in the 'Application Instructions' section, search for it online
        if domain not in email_cache:  # A company with several posts is only searched once
            email_cache[domain] = find_company_emails(browser, company_name, domain, contact_links)
        email_addresses_list = email_cache[domain]
        print(email_addresses_list)
        if email_addresses_list:
            # If emails are found, go to select the most appropriate email address
            best_email = choose_best_email(email_addresses_list)
            print(f"The most appropriate email address is: {best_email}")

    return [company_name, posting['title'], best_email, email_addresses_list, domain,
            posting['date'], posting['link']]


def process_batch(start_page, end_page):
    # Scrape the listing pages start_page to end_page and save every open 'intern' post to the Excel file.
    # Returns the list of records that were collected in this run
    batch_data = []
    failed_links = []
    backup_name = None
    email_cache = {}                            # Web domain -> e-mail addresses found for that company
    seen_links = load_saved_links(OUTPUT_FILE)  # Posts that are already in the Excel file are skipped
    start_time = time.time()

    with BrowserSession() as browser:
        for page_num in range(start_page, end_page + 1):
            rows_seen, postings = get_open_internships(page_num)
            if rows_seen is None:
                print(f"\nCould not load listing page {page_num}, skipping it.")
                continue
            if rows_seen == 0:
                print(f"\nListing page {page_num} has no job posts, so the last page has been reached.")
                break
            print(f"\nListing page {page_num}: {len(postings)} open internship post(s) found")

            for posting in postings:
                if posting['link'] in seen_links:
                    print('\nAlready saved, skipping:', posting['title'])
                    continue
                seen_links.add(posting['link'])
                print('\nPosition title:', posting['title'])

                try:
                    record = build_record(browser, posting, email_cache)
                except Exception as error:  # One bad post must not stop the whole run
                    print(f"  FAILED: {error}. It will be tried again the next time you run this.")
                    failed_links.append(posting['link'])
                    continue

                batch_data.append(record)
                if save_to_excel([record]):  # Save the data to the Excel file
                    print('Record saved')
                else:
                    backup_name = save_backup_csv([record])
                    print(f"Record NOT saved to Excel. It was added to '{backup_name}' instead.")
                time.sleep(0.5)

            # Print status and cumulative time
            elapsed_time = time.time() - start_time
            print(f"Finished processing main listing page {page_num}")
            print(f"Cumulative running time: {elapsed_time:.2f} seconds")

    with_email = sum(1 for record in batch_data if record[2])
    print(f"\nDone. {len(batch_data)} new record(s): {with_email} with an e-mail address, "
          f"{len(batch_data) - with_email} without.")
    if failed_links:
        print(f"{len(failed_links)} post(s) failed and were not saved (run again to retry them):")
        for link in failed_links:
            print('  ', link)
    if backup_name:
        print(f"Some records could not be saved to Excel. Copy them from '{backup_name}'.")
    return batch_data

### 5. Run
Click "Execute Cell". Every record is saved to the Excel file as soon as it is ready,
so nothing is lost if you stop the run. Running it again skips the posts that are already in the file.

In [ ]:
# Measure the start time
start_time = time.time()

# Process the pages (you can adjust START_PAGE and END_PAGE in the settings)
batch_data = process_batch(START_PAGE, END_PAGE)

# Measure the end time
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total running time: {elapsed_time:.2f} seconds")

# A quick look at the records collected in this run
pd.DataFrame(batch_data, columns=COLUMNS)

## Optional
##### You may choose to simply delete all the records that contain nothing in the E-mail column by running this code cell.
To protect your data, nothing is deleted unless you set `DELETE_ROWS_WITHOUT_EMAIL = True` first.

In [ ]:
DELETE_ROWS_WITHOUT_EMAIL = False  # Change to True to delete the rows that have no e-mail address


def delete_empty_emails(file_path):
    # Delete the rows that have nothing in the E-mail column (the colours and column widths are kept)
    workbook = load_workbook(file_path)
    sheet = workbook['Sheet1'] if 'Sheet1' in workbook.sheetnames else workbook.active
    email_column = [cell.value for cell in sheet[1]].index('E-mail') + 1

    deleted = 0
    for row in range(sheet.max_row, 1, -1):  # From the bottom up, so the row numbers stay valid
        value = sheet.cell(row=row, column=email_column).value
        if value is None or not str(value).strip():
            sheet.delete_rows(row)
            deleted += 1

    workbook.save(file_path)
    print(f"Deleted {deleted} row(s) without an e-mail address from {file_path}")


if DELETE_ROWS_WITHOUT_EMAIL:
    delete_empty_emails(OUTPUT_FILE)
else:
    print("Nothing was deleted. Set DELETE_ROWS_WITHOUT_EMAIL = True and run this cell again to delete the rows "
          "without an e-mail address.")